# Fine-tuning de Tipos de veículo

**Oficina Visão Computacional para Cidades Inteligentes**
Semana do Urbanismo de Canaã dos Carajás, 11/11/2026

No Case de tráfego, o modelo pré-treinado contou os Movimentos do cruzamento, mas chamou quase tudo de "carro": a pickup, a van de serviço e o caminhão-plataforma. Ele só conhece as classes com que foi treinado.

Neste notebook, o mesmo modelo aprende cinco **Tipos de veículo**: **Carro**, **Pickup**, **Van**, **Caminhão** e **Ônibus**. Não treinamos do zero: mantemos o **corpo** do modelo, que já sabe ver bordas, partes e objetos, e treinamos só uma **cabeça** nova. Isso é **fine-tuning**.

Na Oficina, o instrutor roda este notebook no projetor. Em casa, ele roda inteiro do mesmo jeito.

| Etapa | O que acontece |
|---|---|
| 0 | Preparar o ambiente |
| 1 | O problema: "quase tudo é carro" |
| 2 | O dataset de Tipos de veículo |
| 3 | Treinar ao vivo, com o corpo congelado |
| 4 | Enquanto treina: anotar e o Moondream |
| 5 | As curvas do treino |
| 6 | O modelo treinado antes |
| 7 | A prova: fotos que o modelo nunca viu |
| 8 | A revanche do "Que veículo é esse?" |
| 9 | O especialista nos 40 segundos |

# 0. Preparar o ambiente

Baixa o código da Oficina, instala os pacotes e baixa o vídeo, o modelo pré-treinado, o dataset e o modelo treinado antes. Leva cerca de 1 minuto e termina com uma checklist ✅.

**Para treinar rápido, use GPU**: menu **Ambiente de execução › Alterar o tipo de ambiente de execução › T4 GPU**, antes de rodar esta célula. Sem GPU, tudo funciona, mas o treino da etapa 3 leva vários minutos por época.

In [ ]:
#@title 0. Preparar o ambiente
import os, sys, subprocess
if "google.colab" in sys.modules:
    PASTA = "/content/oficina-vc-canaa"
    if not os.path.isdir(PASTA + "/.git"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "v2026.11.11",
                        "https://github.com/andrematte/oficina-vc-canaa", PASTA], check=True)
else:
    PASTA = os.path.abspath(".")  # Modo local: o notebook roda de dentro do repositório
if PASTA not in sys.path:
    sys.path.insert(0, PASTA)
from oficina_vc import preparar
preparar("fine_tuning")

# 1. O problema: "quase tudo é carro"

Estes são os recortes do desafio **"Que veículo é esse?"**, tirados do mesmo vídeo de 40 s do Case. Embaixo, a resposta do modelo pré-treinado, o mesmo do Case.

O modelo foi treinado no conjunto COCO, que tem `car`, `truck` e `bus`, mas não tem "pickup" nem "van". Para ele, a pickup e a van só podem ser carro.

In [ ]:
from ultralytics import YOLO
from oficina_vc import caminho, que_veiculo_e_esse

pre_treinado = YOLO(str(caminho("yolo26n.pt")))
que_veiculo_e_esse({"Pré-treinado": pre_treinado});

# 2. O dataset de Tipos de veículo

Para ensinar uma classe nova, o modelo precisa de exemplos com a resposta certa: fotos com uma caixa em volta de cada veículo e o Tipo de veículo escrito nela. Esse conjunto é o **dataset**.

O nosso tem **366 quadros** do mesmo cruzamento de Bellevue, em outros dias e horários (nenhum do trecho de 40 s), divididos em três partes:

- **treino**: as fotos com que o modelo aprende;
- **validação**: fotos para acompanhar o treino, época a época;
- **teste**: fotos guardadas para a prova final, que o modelo nunca vê enquanto aprende.

Repare como o Carro domina: para cada Caminhão, há mais de 30 Carros. As classes raras são as difíceis.

In [ ]:
from oficina_vc import preparar_dataset, contar, amostras

DATASET = preparar_dataset()
contar(DATASET)
amostras(DATASET)

# 3. Treinar ao vivo, com o corpo congelado

O treino começa do modelo pré-treinado do Case, não do zero:

- `freeze=11`: congela as 11 primeiras camadas, o **corpo** (backbone). Elas já sabem ver bordas, partes e objetos, e ficam como estão: são 1,4 milhão dos 2,6 milhões de pesos;
- a **cabeça** (head) é trocada por uma nova, com 5 saídas, uma para cada Tipo de veículo. Ela começa sem saber nada e é a parte que aprende;
- `epochs`: quantas vezes o modelo vê todas as fotos de treino. Cada passada é uma **época**;
- `imgsz=960`: o mesmo tamanho de imagem do Case.

Na Oficina, são só **poucas épocas**, para ver o treino acontecer. O modelo bom, treinado por mais tempo, chega na etapa 6.

In [ ]:
#@title 3. Treinar ao vivo
EPOCAS = 3 #@param {type:"integer"}

from oficina_vc import dispositivo, pasta_do_treino, acompanhar

DISPOSITIVO, nome = dispositivo()
print(f"Treinando em: {nome}")
if DISPOSITIVO == "cpu":
    print("⚠️ Sem GPU, cada época leva vários minutos. Veja a dica da etapa 0.")

modelo = YOLO(str(caminho("yolo26n.pt")))     # o pré-treinado do Case: o ponto de partida
with acompanhar(modelo):                       # uma linha por época, em português
    modelo.train(
        data=str(DATASET),
        epochs=EPOCAS,
        freeze=11,                             # corpo congelado: só a cabeça aprende
        imgsz=960,
        device=DISPOSITIVO,
        project=str(pasta_do_treino().parent),
        name=pasta_do_treino().name,
        exist_ok=True,
        plots=False,
    )

# 4. Enquanto treina: anotar e o Moondream

O treino ocupa o notebook por alguns minutos. Enquanto isso, duas abas do navegador mostram de onde vem um dataset.

**Anotar à mão, no makesense.ai.** Em [makesense.ai](https://www.makesense.ai), carregamos 5 recortes do cruzamento e desenhamos as caixas uma a uma, escolhendo o Tipo de veículo de cada uma. É o trabalho que está por trás de todo modelo treinado: são 5 imagens, e já cansa. O nosso dataset tem 5.291 caixas.

**Anotar com um Modelo fundacional, o Moondream.** No playground do [Moondream](https://moondream.ai), os mesmos recortes são anotados por um modelo de visão e linguagem: perguntamos, em inglês, "is this vehicle a car, a pickup, a van, a truck or a bus?", e ele responde. Um modelo grande e geral (o **Modelo fundacional**) ajuda a montar o dataset que vai treinar um modelo pequeno e rápido (o **Modelo especialista**).

**Como este dataset foi feito**

1. **Quadros**: 366 quadros de 6 vídeos de uma hora do mesmo cruzamento (10 e 11/09/2017), em duas rodadas. A segunda rodada buscou quadros com Pickups, Vans, Caminhões e Ônibus, as classes raras.
2. **Caixas**: o modelo pré-treinado do Case desenhou as caixas. Ele erra a classe, mas acerta onde está o veículo.
3. **Classes**: o Moondream 3 (modelo aberto, licença BSL 1.1), rodando no computador do instrutor, disse o Tipo de veículo de cada caixa.
4. **Revisão**: uma pessoa conferiu as classes em folhas de contato (todos os recortes de uma classe lado a lado) e corrigiu os erros, quase 300 no total.

# 5. As curvas do treino

A cada época, o treino mede o **erro** (loss) nas fotos de validação. É a montanha da teoria: o treino desce o erro, época a época, ajustando os pesos da cabeça.

Ao lado, o **acerto** (mAP50, de 0 a 1; explicado na etapa 7). Com poucas épocas, a descida mal começou.

In [ ]:
from oficina_vc import curvas

curvas(pasta_do_treino(), rotulos=["ao vivo"])

# 6. O modelo treinado antes

Antes da Oficina, o mesmo treino, com o corpo congelado, rodou até o acerto parar de subir (131 épocas, cerca de meia hora): **o modelo treinado antes**, ou **Modelo especialista**. A curva azul é o treino ao vivo; a vermelha, o treino longo.

In [ ]:
from oficina_vc import especialista_pronto

especialista, curvas_do_especialista = especialista_pronto()
print("Classes do especialista:", especialista.names)
curvas(pasta_do_treino(), curvas_do_especialista, rotulos=["ao vivo", "treinado antes"])

# 7. A prova: fotos que o modelo nunca viu

As 64 fotos de **teste** vêm de uma hora que o modelo não viu no treino. Para cada Tipo de veículo:

- **Precisão**: das caixas que o modelo desenhou, quantas estavam certas. Precisão baixa = alarme falso;
- **Revocação**: dos veículos que existiam, quantos o modelo achou. Revocação baixa = veículo esquecido;
- **mAP50**: uma nota de 0% a 100% que junta as duas. Uma caixa conta como certa se cobre pelo menos 50% do veículo e tem a classe certa.

Embaixo, três fotos de teste: à esquerda, a resposta certa; à direita, o que o especialista respondeu.

In [ ]:
from oficina_vc import provar, grade_de_teste

provar(especialista, DATASET, dispositivo=DISPOSITIVO)
grade_de_teste(especialista, DATASET)

# 8. A revanche do "Que veículo é esse?"

Os mesmos recortes da etapa 1, agora com os dois modelos.

In [ ]:
que_veiculo_e_esse({"Pré-treinado": pre_treinado, "Especialista": especialista});

# 9. O especialista nos 40 segundos

Agora o especialista entra no lugar do pré-treinado, no mesmo processamento do Case: as mesmas Zonas por Aproximação, o mesmo rastreador e os mesmos 40 s. A Matriz de movimentos sai separada por Tipo de veículo.

Na GPU, leva menos de um minuto; na CPU, alguns minutos, como no Case.

In [ ]:
from oficina_vc import criar_rastreador, criar_zonas, processar_video, mostrar_video, PASTA_SAIDAS

resultado = processar_video(
    caminho("bellevue-ne8th_2762-2802s.mp4"),
    especialista,
    criar_rastreador(),
    criar_zonas(),
    confianca=0.05,
    imgsz=960,
    passo=2,
    dispositivo=DISPOSITIVO,
    saida=PASTA_SAIDAS / "especialista_saida.mp4",
)
resultado.resumo()
mostrar_video(resultado)

Leia a Matriz com os Tipos de veículo:

1. **O ônibus** (17 s a 21 s, Norte → Sul): o pré-treinado já sabia que era ônibus. E o especialista?
2. **A van de serviço com escada** (3 s a 8 s, Norte → Oeste): de lado, ela parece as minivans, que no dataset contam como Carro. O especialista acertou?
3. **O caminhão-plataforma** (5 s a 10 s, Norte → Leste): siga o número dele na curva. Ele troca de ID? Compare o total de Movimentos com o do Case: o que aconteceu com o Movimento dele?
4. **Do número à decisão**: com a Matriz por Tipo de veículo, o que dá para planejar que a Matriz do Case não dava? Pense em pavimento, vagas de carga e descarga, emissões e linhas de ônibus.

Vans de carga e caminhões-plataforma são raros no dataset. Para o modelo aprender esses dois, o caminho é o mesmo deste notebook: mais fotos deles, anotadas.

**Para continuar em casa**

- Aumente `EPOCAS` na etapa 3 (20 ou mais) e compare as curvas e a prova com o modelo treinado antes.
- Troque `freeze=11` por `freeze=0`, que treina a rede inteira, e veja o que muda no tempo e no acerto.

Para guardar suas mudanças: **Arquivo › Salvar uma cópia no Drive**.